# International generalization — USA / India / Japan

Week4-parity analysis of whether frozen **64-D AlphaEarth** residential vs non-residential
heads transfer across countries.

**Run:** `intl_transfer_20261005_142709`  
**n cells (balanced):** 4,486,790 across 20 cities  
**Protocol:** pure-exclusive clear cells · per-city 1:1 balance (seed 42) · `StandardScaler` on train only · threshold 0.5 · lr64 (+ PCA/PLS probes)

Figures below are pre-rendered from `artifacts/`; re-run the last cells to regenerate.


## 1. Headline

| Within-country LORO (mean F1) | | Key transfers (lr64) | |
|---|---:|---|---:|
| USA | **0.860** | USA → Japan | **0.647** |
| Japan | **0.730** | USA → India | **0.370** |
| India | **0.437** | Japan → USA | **0.722** |

![headline](../figures/01_headline_loro_and_transfer.png)


## 2. Country → country transfer matrix

Diagonal = within-country 80/20 stratified split. Off-diagonal = train on all source-country cities, test on all target-country cities.

![country heat](../figures/02_country_transfer_heatmap.png)

**Read:** USA and Japan have strong within-country ceilings (0.89 / 0.79).
USA→India collapses (0.37); Japan→USA remains competitive (0.72).


## 3. Leave-one-city-out by metro

![loro by city](../figures/03_loro_by_city.png)

USA metros cluster tightly near week4 downtown LORO (~0.78). Japan is mostly strong with Sapporo weaker.
India LORO is near chance for several cities — a national Indian head does not emerge under leave-one-city-out.


## 4. Full city → city transfer heatmap

Blocks outlined for USA (10) / Japan (5) / India (5). Bright = high transfer F1.

![city heat](../figures/04_city_transfer_heatmap.png)


## 5. Block averages & retention

Retention = transfer F1 ÷ source within-country F1 (1.0 = no drop).

![blocks](../figures/05_block_averages_and_retention.png)


## 6. PCA / PLS do not fix cross-country transfer

![models](../figures/06_model_comparison_transfer.png)


## 7. Transfer asymmetry

Points above the diagonal mean B→A works better than A→B.

![asym](../figures/08_transfer_asymmetry.png)


## 8. Sample sizes (analysis extract)

![sizes](../figures/07_sample_sizes.png)


## Takeaways

1. **Embeddings are not globally interchangeable for this task.** Within-USA LORO is excellent; Japan is good; India is fragmented across cities.
2. **A USA-trained head is not safe for India** (F1 ≈ 0.37). Japan is a partial cold-start (≈ 0.65) but still well below USA within-country.
3. **Japan → USA transfers surprisingly well** (≈ 0.72) — asymmetry argues against one universal linear direction.
4. **Operational rule:** train or fine-tune in-country (or at least validate transfer before shipping a US-only head).


## Reproduce / reload tables

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

RUN = Path("../../week4/international_geo_generalization/runs/intl_transfer_20261005_142709")
ART = RUN / "artifacts"
FIG = RUN / "figures"

loro = pd.concat([pd.read_csv(ART / f) for f in ["loro_USA_metrics.csv", "loro_IND_metrics.csv", "loro_JPN_metrics.csv"]], ignore_index=True)
country = pd.read_csv(ART / "country_transfer_metrics.csv")
city = pd.read_csv(ART / "city_transfer_metrics.csv")
summary = json.loads((RUN / "summary.json").read_text())

print("LORO lr64 means:")
print(loro[loro.model=="lr64"].groupby("country")["macro_f1"].agg(["mean","std","min","max"]).round(3))
print("\nCountry transfer lr64:")
print(country[country.model=="lr64"].pivot(index="source_country", columns="target_country", values="macro_f1").round(3))
print("\nFigures:", sorted(p.name for p in FIG.glob("*.png")))


### Optional: regenerate figures

In [ ]:
# Re-run the builder (writes PNGs + can rebuild this notebook)
# !../../.venv/bin/python ../../week4/international_geo_generalization/scripts/build_intl_generalization_notebook.py --run-dir ../../week4/international_geo_generalization/runs/intl_transfer_20261005_142709
from IPython.display import Image, display
for name in sorted(FIG.glob("*.png")):
    print(name.name)
    display(Image(filename=str(name), width=900))
